In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
import gymnasium as gym
from tqdm import tqdm
import optuna

# Force CPU as requested
device = torch.device("cpu")

# Static configurations for training constraints
MIN_REPLAY_SIZE = 10_000
REPLAY_BUFFER_SIZE = 100_000
TOTAL_TIMESTEPS = 300_000  # Fixed budget

OBSERVATION_SPACE = 8
ACTION_SPACE = 2

# --- Define Your Network Architecture ---
class Actor(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(OBSERVATION_SPACE, latent_dim),
            nn.ReLU(),
            nn.Linear(latent_dim, latent_dim),
            nn.ReLU(),
            nn.Linear(latent_dim, ACTION_SPACE),
            nn.Tanh()  # Actions are in range [-1, 1]
        )
    def forward(self, state):
        return self.network(state)

class Critic(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.fc1 = nn.Linear(OBSERVATION_SPACE + ACTION_SPACE, latent_dim)
        self.fc2 = nn.Linear(latent_dim, latent_dim) 
        self.out = nn.Linear(latent_dim, 1)
        self.relu = nn.ReLU()
        
    def forward(self, state, action):
        combined_features = torch.cat([state, action], dim=1)
        x = self.relu(self.fc1(combined_features))
        x = self.relu(self.fc2(x))
        return self.out(x)

# --- Define the Optuna Objective Function ---
def objective(trial):
    # 1. Define the Hyperparameter Search Space
    lr_actor = trial.suggest_float("lr_actor", 1e-5, 1e-3, log=True)
    lr_critic = trial.suggest_float("lr_critic", 1e-4, 5e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [64, 128, 256])
    
    actor_latent_dim = trial.suggest_int("actor_latent_dim", 64, 256, step=64)
    critic_latent_dim = trial.suggest_int("critic_latent_dim", 128, 512, step=128)
    
    tau = trial.suggest_float("tau", 0.001, 0.05, log=True)
    gamma = trial.suggest_float("gamma", 0.95, 0.999)
    
    train_freq = trial.suggest_categorical("train_freq", [1, 2, 4])
    target_update_freq = trial.suggest_categorical("target_update_freq", [1, 5, 10, 20])
    
    sigma_decay = trial.suggest_float("sigma_decay", 0.99, 0.9995)
    sigma_min = trial.suggest_float("sigma_min", 0.01, 0.05)

    # Constants tied to exploration
    SIGMA_START = 1.0

    # 2. Instantiate Networks and Optimizers using Trial Params
    actor = Actor(latent_dim=actor_latent_dim).to(device)
    critic = Critic(latent_dim=critic_latent_dim).to(device)

    actor_target = Actor(latent_dim=actor_latent_dim).to(device)
    critic_target = Critic(latent_dim=critic_latent_dim).to(device)
    actor_target.load_state_dict(actor.state_dict())
    critic_target.load_state_dict(critic.state_dict())

    actor_optimizer = optim.Adam(actor.parameters(), lr=lr_actor)
    critic_optimizer = optim.Adam(critic.parameters(), lr=lr_critic)
    loss_fn = nn.MSELoss()

    replay_buffer = deque(maxlen=REPLAY_BUFFER_SIZE)
    sigma = SIGMA_START

    def update_sigma_decay(s):
        if len(replay_buffer) < MIN_REPLAY_SIZE:
            return s
        return max(sigma_min, s * sigma_decay)

    def train_step():
        mini_batch = random.sample(replay_buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*mini_batch)

        states = torch.FloatTensor(np.array(states)).to(device)
        actions = torch.FloatTensor(np.array(actions)).to(device)
        rewards = torch.FloatTensor(rewards).unsqueeze(1).to(device)
        next_states = torch.FloatTensor(np.array(next_states)).to(device)
        dones = torch.FloatTensor(dones).unsqueeze(1).to(device)

        # Put networks into explicit training mode
        actor.train()
        critic.train()

        # CRITIC UPDATE
        with torch.no_grad():
            next_actions = actor_target(next_states)
            next_q_values = critic_target(next_states, next_actions)
            target_q_values = rewards + (gamma * next_q_values * (1 - dones))

        current_q_values = critic(states, actions)
        critic_loss = loss_fn(current_q_values, target_q_values)
        
        critic_optimizer.zero_grad()
        critic_loss.backward()
        torch.nn.utils.clip_grad_norm_(critic.parameters(), max_norm=1.0) # Added for stability
        critic_optimizer.step()

        # ACTOR UPDATE
        predicted_actions = actor(states)
        actor_loss = -critic(states, predicted_actions).mean()
        
        actor_optimizer.zero_grad()
        actor_loss.backward()
        torch.nn.utils.clip_grad_norm_(actor.parameters(), max_norm=1.0) # Added for stability
        actor_optimizer.step()

    # 3. Environment Loop
    env = gym.make("LunarLander-v3", continuous=True)
    observation, info = env.reset(seed=42)
    
    episode_rewards = []
    current_episode_reward = 0

    for step in range(TOTAL_TIMESTEPS):
        # Warm-up condition check
        if len(replay_buffer) < MIN_REPLAY_SIZE:
            action = env.action_space.sample()
        else:
            actor.eval()
            state_tensor = torch.FloatTensor(observation).unsqueeze(0).to(device)
            with torch.no_grad():
                action = actor(state_tensor).squeeze(0).cpu().numpy()
            
            action = action + np.random.normal(0, sigma, size=ACTION_SPACE)
            action = np.clip(action, -1.0, 1.0)

        next_observation, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
            
        current_episode_reward += reward
        replay_buffer.append((observation, action, reward, next_observation, done))
        observation = next_observation

        # Optimize Main Networks
        if len(replay_buffer) >= MIN_REPLAY_SIZE and step % train_freq == 0:
            train_step()

        # Update Target Networks softly based on trial frequency parameters
        if len(replay_buffer) >= MIN_REPLAY_SIZE and step % target_update_freq == 0:
            with torch.no_grad():
                for param, target_param in zip(actor.parameters(), actor_target.parameters()):
                    target_param.data.copy_(tau * param.data + (1.0 - tau) * target_param.data)
                    
                for param, target_param in zip(critic.parameters(), critic_target.parameters()):
                    target_param.data.copy_(tau * param.data + (1.0 - tau) * target_param.data)

        if done:
            episode_rewards.append(current_episode_reward)
            if len(replay_buffer) >= MIN_REPLAY_SIZE:
                sigma = update_sigma_decay(sigma)

            observation, info = env.reset()
            current_episode_reward = 0

    env.close()
    
    # 4. Return Evaluation Metric
    final_performance = np.mean(episode_rewards[-100:]) if len(episode_rewards) >= 100 else -200.0
    return final_performance

# --- Run the Study ---
if __name__ == "__main__":
    print("🚀 Starting the DDPG optimization process...")
    
    study = optuna.create_study(direction="maximize")
    
    print("Starting the optimization loop now. We will evaluate 50 different configurations...")
    study.optimize(objective, n_trials=50) 
    
    print("\n--- ✨ Optimization Complete! ✨ ---")
    print("Best settings discovered:")
    
    for param_name, param_value in study.best_params.items():
        print(f"  • {param_name}: {param_value}")
        
    print(f"\n🏆 The highest 100-Episode Moving Average Reward achieved was: {study.best_value:.2f}")